# Solar Filament Segmentation — YOLO inference and Kaggle submission

**Francesco Peluso** (`IE22700088`, f.peluso29@studenti.unisa.it)
Machine Learning A.Y. 2025/26 - MSc Computer Engineering, DIEM, University of Salerno
[filament-segmentation-2026](https://www.kaggle.com/competitions/filament-segmentation-2026)

Takes a detector trained by `notebooks/yolo/01_training.ipynb` and writes
`submission.csv`. It trains nothing and tunes nothing: confidence, NMS `iou` and
`max_det` are read from `experiments/<ID>/metrics/submission_recipe.json`, where the
training notebook froze them after selecting them on validation.


In [3]:
import subprocess
import sys
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools", "ultralytics"],
                   check=False)


def locate_project() -> Path:
    """Find the project root from wherever this notebook was opened."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "dataset").is_dir() or (candidate / "experiments").is_dir():
            return candidate
    for candidate in (Path("/content/MachineLearning-Project"),
                      Path("/kaggle/working/MachineLearning-Project")):
        if candidate.is_dir():
            return candidate
    return Path.cwd()


PROJECT_ROOT = locate_project()

# The competition zip's own layout nests everything one level deeper
# (dataset/MAGFiLO_1.0_Kaggle_2026/train/...) than a plain "dataset/train/..." tree
# does. Both are accepted below, whichever a local copy or a kagglehub download hands
# back, so no extra path-fixing cell is ever needed.
def _has_dataset(root: Path) -> bool:
    return (root / "train").is_dir() or (root / "MAGFiLO_1.0_Kaggle_2026").is_dir()


DATASET_DIR = PROJECT_ROOT / "dataset"

if ON_COLAB and not _has_dataset(DATASET_DIR):
    # Nothing local yet (no Drive mount, no manual unzip): pull the competition data
    # straight from Kaggle. This needs Kaggle credentials available in the session
    # (Colab Secrets KAGGLE_USERNAME/KAGGLE_KEY, a kaggle.json, or kagglehub's own
    # interactive login) and the competition rules accepted on kaggle.com - if either
    # is missing, kagglehub raises its own clear error here rather than failing later
    # with a bare FileNotFoundError deep inside data loading.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=False)
    import kagglehub
    DATASET_DIR = Path(kagglehub.competition_download("filament-segmentation-2026"))
    print(f"downloaded competition data to {DATASET_DIR}")

if not (DATASET_DIR / "train").is_dir() and (DATASET_DIR / "MAGFiLO_1.0_Kaggle_2026").is_dir():
    DATASET_DIR = DATASET_DIR / "MAGFiLO_1.0_Kaggle_2026"
TRAIN_IMAGES = DATASET_DIR / "train" / "train_images"
ANNOTATIONS = DATASET_DIR / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
TEST_IMAGES = DATASET_DIR / "test" / "test_images"

print(f"project : {PROJECT_ROOT}")
print(f"dataset : {DATASET_DIR}  (exists: {DATASET_DIR.is_dir()})")
if not ANNOTATIONS.is_file():
    listing = sorted(p.name for p in DATASET_DIR.iterdir()) if DATASET_DIR.is_dir() else []
    print(f"WARNING: annotations not found at {ANNOTATIONS}")
    print(f"  contents of {DATASET_DIR}: {listing}")

project : /Users/fp/GitHub Repos/MachineLearning-Project
dataset : /Users/fp/GitHub Repos/MachineLearning-Project/dataset  (exists: True)


## 1. Detector and frozen recipe

In [4]:
import csv
import json
import time

import numpy as np
import torch
from PIL import Image
from pycocotools import mask as mask_util
from tqdm.auto import tqdm
from ultralytics import YOLO

EXPERIMENT_ID = "Y2"
SUBMISSION_PATH = PROJECT_ROOT / "submission.csv"
NATIVE = 2048

EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / EXPERIMENT_ID
recipe = json.loads((EXPERIMENT_DIR / "metrics" / "submission_recipe.json").read_text())
print(json.dumps(recipe, indent=2))

device = (0 if torch.cuda.is_available() else
          "mps" if getattr(torch.backends, "mps", None) is not None
          and torch.backends.mps.is_available() else "cpu")
yolo = YOLO(str(PROJECT_ROOT / recipe["checkpoint"]))
print(f"\n{recipe['detector']} loaded on {device}")

{
  "experiment_id": "Y1",
  "detector": "yolov8m-seg.pt",
  "checkpoint": "experiments/Y1/checkpoints/best_model.pt",
  "image_size": 1792,
  "conf": 0.35,
  "iou": 0.0,
  "max_det": 50,
  "retina_masks": true,
  "validation_dice": 0.6150244196411391,
  "validation_pq": 0.41696314969028375
}

yolov8m-seg.pt loaded on mps


## 2. From detections to CSV rows

For each test image: one detector pass at the recipe's resolution, candidates kept
above the recipe's confidence and capped at `max_det`, overlapping masks made disjoint
in confidence order (the scorer expects one instance per pixel), and each instance
encoded as COCO run-length encoding at 2048x2048.


In [5]:
def predict_instances(path):
    """Disjoint instance masks at native resolution, strongest first."""
    result = yolo.predict(str(path), imgsz=recipe["image_size"], conf=recipe["conf"],
                          iou=recipe["iou"], max_det=recipe["max_det"],
                          retina_masks=recipe.get("retina_masks", True),
                          device=device, verbose=False)[0]
    if result.masks is None:
        return []
    masks = result.masks.data.cpu().numpy().astype(np.uint8)
    confs = result.boxes.conf.cpu().numpy()
    taken = np.zeros((NATIVE, NATIVE), dtype=bool)
    instances = []
    for index in np.argsort(-confs):
        mask = masks[index]
        if mask.shape != (NATIVE, NATIVE):
            mask = np.asarray(Image.fromarray(mask).resize((NATIVE, NATIVE),
                                                           Image.Resampling.NEAREST))
        clean = mask.astype(bool) & ~taken
        if clean.sum() < 5:
            continue
        taken |= clean
        instances.append(clean.astype(np.uint8))
    return instances


def rle_counts(mask):
    return mask_util.encode(np.asfortranarray(mask))["counts"].decode("ascii")


test_paths = sorted(TEST_IMAGES.glob("*.jpeg"))
print(f"{len(test_paths)} official test images")
started = time.perf_counter()
demo = predict_instances(test_paths[0])
print(f"{test_paths[0].name}: {len(demo)} filaments, {time.perf_counter() - started:.2f}s")

180 official test images
20110120105534Ch.jpeg: 1 filaments, 3.05s


## 3. The whole test set

In [6]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
rows, per_image = [], []
for path in tqdm(test_paths, desc="test images"):
    instances = predict_instances(path)
    per_image.append(len(instances))
    for index, instance in enumerate(instances, start=1):
        rows.append((f"{path.stem}_{index}", rle_counts(instance)))

with SUBMISSION_PATH.open("w", encoding="utf-8", newline="") as handle:
    handle.write("filament_id,segmentation_rle\n")
    for filament_id, encoded in rows:
        handle.write(f"{filament_id},{encoded}\n")

stats = {"path": str(SUBMISSION_PATH), "images": len(per_image), "filaments": len(rows),
         "filaments_per_image": float(np.mean(per_image)),
         "images_without_filaments": int(sum(1 for c in per_image if c == 0)),
         "seconds": time.perf_counter() - started,
         "gpu_memory_gb": (torch.cuda.max_memory_allocated() / 1024 ** 3
                           if torch.cuda.is_available() else float("nan")),
         "recipe": recipe}
(EXPERIMENT_DIR / "metrics" / "submission_stats.json").write_text(
    json.dumps(stats, indent=2, default=float), encoding="utf-8")
print(json.dumps(stats, indent=2, default=float))

test images: 100%|██████████| 180/180 [00:42<00:00,  4.28it/s]

{
  "path": "/Users/fp/GitHub Repos/MachineLearning-Project/submission.csv",
  "images": 180,
  "filaments": 1085,
  "filaments_per_image": 6.027777777777778,
  "images_without_filaments": 2,
  "seconds": 42.07711754200864,
  "gpu_memory_gb": NaN,
  "recipe": {
    "experiment_id": "Y1",
    "detector": "yolov8m-seg.pt",
    "checkpoint": "experiments/Y1/checkpoints/best_model.pt",
    "image_size": 1792,
    "conf": 0.35,
    "iou": 0.0,
    "max_det": 50,
    "retina_masks": true,
    "validation_dice": 0.6150244196411391,
    "validation_pq": 0.41696314969028375
  }
}


## 4. Sanity checks before uploading

A malformed submission scores zero, and the feedback loop on Kaggle is slow. These
checks cost a second and catch the mistakes that actually happen: a missing header,
duplicate ids, an RLE that does not decode, a mask at the wrong resolution.

In [7]:
lines = SUBMISSION_PATH.read_text().splitlines()
header, body = lines[0], lines[1:]
assert header == "filament_id,segmentation_rle", f"unexpected header: {header}"
ids = [line.split(",", 1)[0] for line in body]
assert len(ids) == len(set(ids)), "duplicate filament ids"
stems = {i.rsplit("_", 1)[0] for i in ids}
assert stems <= {p.stem for p in test_paths}, "ids that match no test image"

areas = []
for line in body[::40]:
    decoded = mask_util.decode({"size": [NATIVE, NATIVE],
                                "counts": line.split(",", 1)[1].encode("ascii")})
    assert decoded.shape == (NATIVE, NATIVE)
    areas.append(int(decoded.sum()))
assert min(areas) > 0, "an RLE decodes to an empty mask"

print(f"rows                    : {len(body)}")
print(f"images represented      : {len(stems)} / {len(test_paths)}")
print(f"filaments per image     : {stats['filaments_per_image']:.1f}")
print(f"decoded areas (sampled) : min {min(areas)}, median {int(np.median(areas))}, max {max(areas)}")
print(f"\nOK -> upload {SUBMISSION_PATH.name} "
      f"(validation Dice {recipe['validation_dice']:.3f} / PQ {recipe['validation_pq']:.3f})")

rows                    : 1085
images represented      : 178 / 180
filaments per image     : 6.0
decoded areas (sampled) : min 375, median 1582, max 15020

OK -> upload submission.csv (validation Dice 0.615 / PQ 0.417)
